# Data and Feature Preparation
PaySim synthetic dataset: loading → EDA → filtering → feature engineering → three buckets → train/val/test split → parquet export.

## 1.1 Loading and Sanity Checks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

DATA_DIR = os.path.join("..", "data")
CSV_PATH = os.path.join(DATA_DIR, "PS_20174392719_1491204439457_log.csv")

# The data is large (~6.3M rows, ~470 MB); load the full file first,
# uncomment the line below with nrows=500_000 to prototype if memory is tight.
df = pd.read_csv(CSV_PATH)  # nrows=500_000

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
print("--- Data Types ---")
print(df.dtypes)
print("\n--- Missing Values ---")
print(df.isna().sum())

## 1.2 Exploratory Analysis (EDA)

In [ ]:
# Overall fraud rate (expected ~0.13%)
fraud_rate = df["isFraud"].mean()
fraud_count = df["isFraud"].sum()
print(f"Fraud rate : {fraud_rate:.4%}")
print(f"Fraud count: {fraud_count:,} / {len(df):,}")

In [ ]:
# type × isFraud cross-tabulation
cross = pd.crosstab(df["type"], df["isFraud"], margins=True)
cross.columns = ["Clean", "Fraud", "Total"]
cross["Fraud Rate"] = (cross["Fraud"] / cross["Total"]).map("{:.4%}".format)
print(cross)

# Sanity check: fraud should only appear in TRANSFER and CASH_OUT
fraud_by_type = df[df["isFraud"] == 1]["type"].value_counts()
print("\nTransaction types with fraud:")
print(fraud_by_type)
assert set(fraud_by_type.index).issubset({"TRANSFER", "CASH_OUT"}), \
    "Fraud found in other types too - unexpected!"
print("\nFraud only occurs in TRANSFER and CASH_OUT, as expected.")

In [ ]:
# Amount distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(np.log1p(df["amount"]), bins=60, edgecolor="none", color="steelblue", alpha=0.7)
axes[0].set_title("log(1+amount) Distribution")
axes[0].set_xlabel("log(1+amount)")
axes[0].set_ylabel("Frequency")

fraud_amounts = np.log1p(df[df["isFraud"] == 1]["amount"])
clean_amounts = np.log1p(df[df["isFraud"] == 0]["amount"])
axes[1].hist(clean_amounts, bins=60, alpha=0.6, label="Clean", color="steelblue")
axes[1].hist(fraud_amounts, bins=60, alpha=0.7, label="Fraud", color="tomato")
axes[1].set_title("Fraud vs Clean - Amount Distribution")
axes[1].set_xlabel("log(1+amount)")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join("..", "reports", "amount_dist.png"), dpi=120)
plt.show()
print("Chart saved: reports/amount_dist.png")

In [ ]:
# Zero-balance account rate
zero_orig = (df["oldbalanceOrg"] == 0).mean()
zero_dest = (df["oldbalanceDest"] == 0).mean()
print(f"oldbalanceOrg == 0 rate : {zero_orig:.4%}")
print(f"oldbalanceDest == 0 rate: {zero_dest:.4%}")

In [ ]:
# Overlap between isFlaggedFraud and real fraud (understand why we drop it)
flagged = df["isFlaggedFraud"].sum()
real_fraud = df["isFraud"].sum()
both = (df["isFlaggedFraud"] & df["isFraud"]).sum()
flagged_not_fraud = (df["isFlaggedFraud"] & (df["isFraud"] == 0)).sum()
fraud_not_flagged = ((df["isFlaggedFraud"] == 0) & df["isFraud"]).sum()

print(f"isFlaggedFraud == 1     : {flagged:,}")
print(f"isFraud == 1            : {real_fraud:,}")
print(f"Both are 1 (overlap)    : {both:,}")
print(f"Flagged but not fraud   : {flagged_not_fraud:,}  ← noise")
print(f"Fraud but not flagged   : {fraud_not_flagged:,}  ← missed")
print()
print("Conclusion: isFlaggedFraud is a nearly useless rule and carries leakage risk,")
print("            so it is dropped and never enters the rule engine.")

## 1.3 Filtering and Leakage Prevention

In [ ]:
# TRANSFER + CASH_OUT only
df = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
print(f"After filtering: {df.shape}")
print(f"Fraud rate (subset): {df['isFraud'].mean():.4%}")

# Drop isFlaggedFraud - a leakage source, never enters the model/rules
df.drop(columns=["isFlaggedFraud"], inplace=True)
print("\nisFlaggedFraud column dropped.")
print("Remaining columns:", df.columns.tolist())

# nameOrig / nameDest: identity leakage risk; kept in the dataframe
# but never added to ML_FEATURES or RULE_FIELDS.
# May be retained later for UI/LLM context.
print("\nnameOrig/nameDest present in the dataframe (won't enter the model):")
print(df[["nameOrig", "nameDest"]].head(3))

## 1.4 Feature Engineering

In [ ]:
# Feature formulas
df["errorBalanceOrig"] = df["newbalanceOrig"] + df["amount"] - df["oldbalanceOrg"]
df["errorBalanceDest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
df["step_hour"]        = df["step"] % 24
df["is_transfer"]      = (df["type"] == "TRANSFER").astype(int)
df["is_cashout"]       = (df["type"] == "CASH_OUT").astype(int)

print("New columns added:", ["errorBalanceOrig","errorBalanceDest","step_hour","is_transfer","is_cashout"])

In [ ]:
# Manual check: errorBalance should be ≈ 0 for a consistent transaction
# and deviate from zero for inconsistent (fraud) ones.
sample_ok   = df[(df["errorBalanceOrig"].abs() < 1)].head(3)
sample_bad  = df[(df["errorBalanceOrig"].abs() > 1000)].head(3)

print("Consistent transactions (errorBalanceOrig ≈ 0):")
print(sample_ok[["amount","oldbalanceOrg","newbalanceOrig","errorBalanceOrig","isFraud"]])
print("\nInconsistent transactions (large errorBalanceOrig):")
print(sample_bad[["amount","oldbalanceOrg","newbalanceOrig","errorBalanceOrig","isFraud"]])

# errorBalanceOrig stats for fraud transactions
print("\nerrorBalanceOrig stats for fraud transactions:")
print(df[df["isFraud"]==1]["errorBalanceOrig"].describe())
print("\nerrorBalanceOrig stats for clean transactions:")
print(df[df["isFraud"]==0]["errorBalanceOrig"].describe())

In [ ]:
# Generate synthetic fields independent of the label (UI / LLM enrichment; never enters the model)
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
n   = len(df)

df["device_id"]         = rng.integers(10_000, 99_999, n).astype(str)
df["is_known_device"]   = rng.choice([0, 1], n, p=[0.3, 0.7])
df["login_country"]     = rng.choice(["TR","DE","NL","US","GB"], n, p=[.70,.10,.08,.07,.05])
df["geo_velocity_flag"] = rng.choice([0, 1], n, p=[0.9, 0.1])
df["channel"]           = rng.choice(["mobile","web","atm","branch"], n, p=[.55,.25,.15,.05])

# Confirm independence from the label (randomly generated → |corr| ≈ 0)
for col in ["is_known_device", "geo_velocity_flag"]:
    corr = round(df[col].corr(df["isFraud"]), 4)
    print(f"corr({col}, isFraud): {corr}")

## 1.5 Three Buckets (Leakage Wall)

In [ ]:
# Raw data → rule engine only
RULE_FIELDS = [
    "amount", "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest", "type", "step"
]

# Derived → ML model only
ML_FEATURES = [
    "amount", "step_hour", "errorBalanceOrig",
    "errorBalanceDest", "is_transfer", "is_cashout"
]

# Synthetic → UI / LLM only (NEVER enters the model, score, rules, or automation)
SYNTHETIC_FIELDS = [
    "device_id", "is_known_device", "login_country",
    "geo_velocity_flag", "channel"
]

TARGET = "isFraud"

# 1) List intersection check
leak = set(ML_FEATURES) & set(SYNTHETIC_FIELDS)
assert len(leak) == 0, f"LEAK! Synthetic field in ML_FEATURES: {leak}"
leak2 = set(RULE_FIELDS) & set(SYNTHETIC_FIELDS)
assert len(leak2) == 0, f"LEAK! Synthetic field in RULE_FIELDS: {leak2}"

# 2) Confirm the synthetic fields actually exist in df
for sf in SYNTHETIC_FIELDS:
    assert sf in df.columns, f"Synthetic field missing from df: {sf}"

print("RULE_FIELDS  :", RULE_FIELDS)
print("ML_FEATURES  :", ML_FEATURES)
print("SYNTHETIC    :", SYNTHETIC_FIELDS)
print("\nNo overlap between the three buckets.")

## 1.6 Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

X = df[ML_FEATURES]
y = df[TARGET]

# 60 / 20 / 20 stratified split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train : {X_train.shape}  |  fraud: {y_train.mean():.4%}")
print(f"Val   : {X_val.shape}    |  fraud: {y_val.mean():.4%}")
print(f"Test  : {X_test.shape}   |  fraud: {y_test.mean():.4%}")

In [ ]:
# Scaler - fit ONLY on train (leakage prevention)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=ML_FEATURES, index=X_train.index
)
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val), columns=ML_FEATURES, index=X_val.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=ML_FEATURES, index=X_test.index
)

print("Scaler fit completed on train only.")
print("Scaled train mean (should be ≈0):", X_train_scaled.mean().round(6).to_dict())

In [ ]:
# Save the processed sets as parquet
import joblib

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(os.path.join("..", "models"), exist_ok=True)

# All columns (raw + derived + synthetic + target)
# → the rule engine, UI, and LLM read from these parquet files; the model selects df[ML_FEATURES]
df.loc[X_train.index].to_parquet(os.path.join(DATA_DIR, "train.parquet"), index=False)
df.loc[X_val.index].to_parquet(os.path.join(DATA_DIR, "val.parquet"), index=False)
df.loc[X_test.index].to_parquet(os.path.join(DATA_DIR, "test.parquet"), index=False)

# Scaled sets - ML_FEATURES + isFraud only (for LogReg)
X_train_scaled.assign(isFraud=y_train).to_parquet(
    os.path.join(DATA_DIR, "train_scaled.parquet"), index=False
)
X_val_scaled.assign(isFraud=y_val).to_parquet(
    os.path.join(DATA_DIR, "val_scaled.parquet"), index=False
)
X_test_scaled.assign(isFraud=y_test).to_parquet(
    os.path.join(DATA_DIR, "test_scaled.parquet"), index=False
)

# Save the scaler
joblib.dump(scaler, os.path.join("..", "models", "scaler_v1.pkl"))

print("Saved files:")
for f in ["train","val","test","train_scaled","val_scaled","test_scaled"]:
    path = os.path.join(DATA_DIR, f"{f}.parquet")
    size_mb = os.path.getsize(path) / 1_048_576
    print(f"  data/{f}.parquet  ({size_mb:.1f} MB)")
print("  models/scaler_v1.pkl")

# Content check
train_full = pd.read_parquet(os.path.join(DATA_DIR, "train.parquet"))
print(f"\ntrain.parquet columns ({len(train_full.columns)}):", train_full.columns.tolist())
assert set(ML_FEATURES).issubset(train_full.columns), "ML_FEATURES missing!"
assert set(RULE_FIELDS).issubset(train_full.columns), "RULE_FIELDS missing!"
assert set(SYNTHETIC_FIELDS).issubset(train_full.columns), "SYNTHETIC_FIELDS missing!"

## Summary

In [ ]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  Filtered data     : {df.shape[0]:,} rows (TRANSFER + CASH_OUT)")
print(f"  Total columns     : {df.shape[1]} (raw + derived + synthetic + target)")
print(f"  Fraud rate        : {df['isFraud'].mean():.4%}")
print(f"  ML_FEATURES       : {ML_FEATURES}")
print(f"  Split             : 60/20/20 stratified (seed={RANDOM_STATE})")
print(f"  Scaler            : fit=train only → no leakage")
print(f"  isFlaggedFraud    : dropped")
print(f"  errorBalance      : formula verified")
print(f"  Synthetic fields  : present in df, never enter model/score/rules (verified)")
print( "  train/val/test    : all columns (for the rule engine + UI/LLM)")
print( "  *_scaled parquet  : ML_FEATURES + isFraud only (for LogReg)")
print("=" * 60)